# ABN（CIFAR100を用いたCNNの可視化）

---
## 目的
Attention Branch Network (ABN) の構造を理解する．ABNを用いて，CIFAR100データセットに対する画像認識と，その判断根拠（Attention Map）の可視化を同時に行う．

## Attention Branch Network (ABN)
`cam.ipynb`で扱ったCAMは，学習済みのネットワークに対して事後的にAttention Mapを計算する手法でした．一方，ABN[1]は，Attention機構をネットワークの構造そのものに組み込み，学習時にAttention Mapを直接獲得しながら認識精度の向上も図る手法です．

ABNは，次の3つのモジュールから構成されます．

1. **Feature Extractor**：入力画像から特徴マップを抽出する，Attention BranchとPerception Branchで共有される部分．
2. **Attention Branch**：Feature Extractorの出力から，CAMと同様の仕組み（クラスごとの特徴マップをGAPし，そのクラスらしさを学習する）で，1チャンネルのAttention Map $M(x)$を生成する．
3. **Perception Branch**：Feature Extractorの出力にAttention Mapを重み付けした特徴 $g(x) = x \cdot (1 + M(x))$ を入力として，最終的なクラス分類を行う．

Attention Branchは学習の過程でCAMに似た機構により**教師ラベルを用いて直接学習される**ため，学習が進むほどAttention Mapが認識に有効な領域を捉えるようになります．さらに，そのAttention Mapが持つ「注視領域を強調する」効果をPerception Branchの入力に反映させることで，認識精度の向上も期待できます．

[1] H. Fukui et al., "Attention Branch Network: Learning of Attention Mechanism for Visual Explanation," CVPR, 2019.

## モジュールのインポート
はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
from time import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
CIFAR100データセットを読み込みます．データセットの詳細については`classification/resnet.ipynb`を参照してください．

In [ ]:
transform_train = transforms.Compose([transforms.RandomCrop(32, padding=4),
                                       transforms.RandomHorizontalFlip(),
                                       transforms.ToTensor()])
transform_test = transforms.Compose([transforms.ToTensor()])

train_data = torchvision.datasets.CIFAR100(root="./", train=True, transform=transform_train, download=True)
test_data = torchvision.datasets.CIFAR100(root="./", train=False, transform=transform_test, download=True)
class_names = train_data.classes

print(len(train_data), len(test_data))

## Feature Extractor
`classification/resnet.ipynb`と同じBasicBlockを用いて，Attention BranchとPerception Branchで共有するFeature Extractorを構築します．CIFAR版ResNet（depth=20）の最初の2つのブロック（`layer1`, `layer2`）に相当する部分までを共有し，そこから先をAttention BranchとPerception Branchに分岐させます．

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super().__init__()
        self.convs = nn.Sequential(
            nn.Conv2d(inplanes, planes, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(planes),
            nn.ReLU(inplace=True),
            nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(planes),
        )
        self.downsample = downsample
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = x if self.downsample is None else self.downsample(x)
        out = self.convs(x)
        out += residual
        return self.relu(out)


def make_layer(inplanes, planes, n_blocks, stride=1):
    """BasicBlockをn_blocks個積んだResidual Blockの列を作成する（resnet.ipynbの_make_layerと同様）"""
    downsample = None
    if stride != 1 or inplanes != planes * BasicBlock.expansion:
        downsample = nn.Sequential(
            nn.Conv2d(inplanes, planes * BasicBlock.expansion, kernel_size=1, stride=stride, bias=False),
            nn.BatchNorm2d(planes * BasicBlock.expansion),
        )
    layers = [BasicBlock(inplanes, planes, stride, downsample)]
    inplanes = planes * BasicBlock.expansion
    for _ in range(n_blocks - 1):
        layers.append(BasicBlock(inplanes, planes))
    return nn.Sequential(*layers), inplanes


class FeatureExtractor(nn.Module):
    def __init__(self, n_blocks=3):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.relu = nn.ReLU(inplace=True)

        self.layer1, inplanes = make_layer(16, 16, n_blocks)
        self.layer2, inplanes = make_layer(inplanes, 32, n_blocks, stride=2)
        self.out_channels = inplanes  # 32

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)  # (N, 32, 16, 16)
        return x

## Attention Branch
Feature Extractorの出力（(N, 32, 16, 16)）を入力とし，2つの出力を計算します．

1. **Attention Branchの分類出力**：CAMと同じ仕組みで，Residual Blockに続けてクラス数チャンネルの特徴マップ（class activation maps）を1x1畳み込みで作り，GAPしてクラスごとのスコアとする．学習時はこの出力に対する交差エントロピー損失（Attention Loss）により，特徴マップがクラスごとに意味のある応答を持つよう学習される．
2. **Attention Map $M(x)$**：上記のクラスごとの特徴マップを，1x1畳み込みで1チャンネルに集約し，Batch NormalizationとSigmoidを適用することで，0〜1の範囲を持つ1チャンネルのAttention Mapを得る．

Attention Branch内のResidual Blockは，Perception Branchとの特徴マップサイズを合わせるため，`stride=1`としてダウンサンプリングを行いません．

In [ ]:
class AttentionBranch(nn.Module):
    def __init__(self, in_channels, n_class=100, n_blocks=3):
        super().__init__()
        self.layer, out_channels = make_layer(in_channels, 64, n_blocks, stride=1)  # (N, 64, 16, 16)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.conv_class = nn.Conv2d(out_channels, n_class, kernel_size=1)  # クラスごとのclass activation map
        self.avgpool = nn.AdaptiveAvgPool2d(1)

        self.conv_attention = nn.Conv2d(n_class, 1, kernel_size=1)  # class activation mapを1チャンネルに集約
        self.bn_attention = nn.BatchNorm2d(1)

    def forward(self, x):
        x = self.relu(self.bn(self.layer(x)))
        class_maps = self.conv_class(x)  # (N, n_class, 16, 16)

        att_output = self.avgpool(class_maps).flatten(1)  # (N, n_class)：Attention Branch自体の分類出力

        attention_map = torch.sigmoid(self.bn_attention(self.conv_attention(class_maps)))  # (N, 1, 16, 16)
        return att_output, attention_map

## Perception Branch
Feature Extractorの出力に，Attention Branchが生成したAttention Map $M(x)$を用いて重み付けを行います．

$$g(x) = x \cdot (1 + M(x))$$

「$1 + M(x)$」倍することで，Attention Mapの値が大きい（注視すべき）領域の特徴を強調しつつ，Attention Mapが0であっても元の特徴量$x$自体は保持される（完全に0にはしない）ようにしています．この$g(x)$を入力として，残りのResidual Blockを経てGAP・全結合層により最終的なクラス分類を行います．

In [ ]:
class PerceptionBranch(nn.Module):
    def __init__(self, in_channels, n_class=100, n_blocks=3):
        super().__init__()
        self.layer, out_channels = make_layer(in_channels, 64, n_blocks, stride=2)  # (N, 64, 8, 8)
        self.avgpool = nn.AvgPool2d(8)
        self.fc = nn.Linear(out_channels, n_class)

    def forward(self, x, attention_map):
        g_x = x * (1 + attention_map)  # Attention Mapによる特徴の重み付け
        x = self.layer(g_x)
        x = self.avgpool(x)
        x = x.flatten(1)
        return self.fc(x)


class ABN(nn.Module):
    def __init__(self, n_class=100, n_blocks=3):
        super().__init__()
        self.feature_extractor = FeatureExtractor(n_blocks)
        self.attention_branch = AttentionBranch(self.feature_extractor.out_channels, n_class, n_blocks)
        self.perception_branch = PerceptionBranch(self.feature_extractor.out_channels, n_class, n_blocks)

    def forward(self, x):
        feat = self.feature_extractor(x)
        att_output, attention_map = self.attention_branch(feat)
        per_output = self.perception_branch(feat, attention_map)
        return att_output, per_output, attention_map

## 損失関数
ABNの損失は，Attention Branchの分類出力に対する損失（Attention Loss）と，Perception Branchの最終的な分類出力に対する損失（Perception Loss）の和で構成されます．どちらも通常のクラス分類と同じ交差エントロピー損失を用います．

$$L = L_{att}(\text{att\_output}, y) + L_{per}(\text{per\_output}, y)$$

In [ ]:
criterion = nn.CrossEntropyLoss().to(device)

## ネットワークの作成
ABNのネットワークを作成します．Attention BranchとPerception Branchの2つの出力を持つ多入出力モデルであるため，`torchsummary`は使用せずパラメータ数のみを表示します．最適化手法・学習率などは`resnet.ipynb`と同じ設定とします．

In [ ]:
model = ABN(n_class=100, n_blocks=3).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)

num_params = sum(p.numel() for p in model.parameters())
print('パラメータ数:', num_params)

## 学習
CIFAR100データセットを用いて学習します．ミニバッチサイズを128，学習エポック数を20とします．

In [ ]:
batch_size = 128
epoch_num = 20

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2)

model.train()
train_start = time()
for epoch in range(1, epoch_num + 1):
    sum_loss = 0.0
    count = 0

    for image, label in train_loader:
        image = image.to(device)
        label = label.to(device)

        att_output, per_output, _ = model(image)
        loss = criterion(att_output, label) + criterion(per_output, label)

        model.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()
        pred = torch.argmax(per_output, dim=1)  # 最終的な認識結果はPerception Branchの出力を使用
        count += torch.sum(pred == label)

    print(f'epoch: {epoch}, mean loss: {sum_loss / len(train_loader):.4f}, mean accuracy: {count.item() / len(train_data):.4f}, elapsed_time: {time() - train_start:.4f}')

## テスト
学習したネットワークモデルを用いて評価を行います．最終的な認識結果にはPerception Branchの出力を使用します．

In [ ]:
test_loader = torch.utils.data.DataLoader(test_data, batch_size=100, shuffle=False)

model.eval()
count = 0
with torch.no_grad():
    for image, label in test_loader:
        image = image.to(device)
        label = label.to(device)

        _, per_output, _ = model(image)

        pred = torch.argmax(per_output, dim=1)
        count += torch.sum(pred == label)

print("test accuracy: {}".format(count.item() / len(test_data)))

## Attention Mapの可視化
評価用データからいくつかの画像を取り出し，Attention Branchが出力するAttention Mapを可視化します．CAMとは異なり，Attention Mapはネットワークの推論（1回のforward）だけで得られる点に注目してください．

In [ ]:
model.eval()
n_show = 6
fig, axes = plt.subplots(2, n_show, figsize=(2.2 * n_show, 4.6))

for i in range(n_show):
    image, label = test_data[i]
    image_batch = image.unsqueeze(0).to(device)

    with torch.no_grad():
        _, per_output, attention_map = model(image_batch)
    pred_class = torch.argmax(per_output, dim=1).item()

    attention_map_resized = F.interpolate(attention_map, size=32, mode='bilinear', align_corners=False)
    attention_map_np = attention_map_resized.squeeze().cpu().numpy()

    image_np = image.permute(1, 2, 0).numpy()
    heatmap = plt.get_cmap('jet')(attention_map_np)[:, :, :3]
    overlay = 0.5 * image_np + 0.5 * heatmap

    axes[0, i].imshow(image_np); axes[0, i].axis('off')
    axes[0, i].set_title(f'GT: {class_names[label]}', fontsize=9)
    axes[1, i].imshow(overlay); axes[1, i].axis('off')
    axes[1, i].set_title(f'pred: {class_names[pred_class]}', fontsize=9)

plt.tight_layout()
plt.show()

## オリジナルのABNとの違い

| 項目 | オリジナル論文 (Fukui et al., 2019) | 本ノートブック |
| :-- | :-- | :-- |
| バックボーン | ResNet-family（ResNet-110やWide ResNetなど，ImageNet版はResNet-152も使用） | CIFAR版ResNet（`resnet.ipynb`と同じdepth=20相当）を簡略化して使用 |
| データセット | CIFAR10/100，ImageNet，複数のFine-grained認識データセットなど | CIFAR100のみ |
| Attention機構の適用回数 | Attention Branch内のResidual Blockでも段階的にAttentionを適用する構成が提案されている | Perception Branchの入力時に1回のみ適用 |
| データ拡張 | Random Erasingなど複数の工夫を実施 | `RandomCrop`, `RandomHorizontalFlip`のみ |

## 課題

1. `cam.ipynb`で可視化したCAMと，本ノートブックのAttention Mapを同じ画像で比較し，着目領域がどのように異なるか確認してください．
2. Attention Lossの重みを変更（例えば`criterion(att_output, label) * 0.5`のように調整）し，認識精度やAttention Mapの見え方がどのように変化するか確認してください．
3. Perception Branchで`g(x) = x * (1 + M(x))`の代わりに`g(x) = x * M(x)`（Attention Mapが0の領域を完全に無視する）を使った場合，学習の安定性や認識精度にどのような影響があるか調べてください．